# 🎬 Viral Video 9:16 Short Cloud Renderer (Google Colab 12GB RAM)

This notebook turns Google Colab into a high-speed **FFmpeg & yt-dlp video worker** for **[viral.mrpallanti.in](https://viral.mrpallanti.in/)**.

### Instructions:
1. Click the **Play button ▶️** on the code cell below.
2. Wait ~20 seconds until you see the **Cloudflare Tunnel URL** (e.g. `https://xxxx.trycloudflare.com`).
3. Copy that URL and paste it into the **Colab Worker URL** box in your ViralClip Studio!
4. Keep this tab open while you create your shorts.

In [ ]:
# 1. Install dependencies & Cloudflare tunnel
!pip install -q fastapi uvicorn yt-dlp pydantic
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# 2. Create the FastAPI worker server code
server_code = """import os, uuid, subprocess, re
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel

app = FastAPI(title="Colab Video Engine")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

OUTPUT_DIR = "/content/exports"
os.makedirs(OUTPUT_DIR, exist_ok=True)

class RenderRequest(BaseModel):
    url: str
    start_seconds: float = 0
    end_seconds: float = 30
    hook_text: str = ""

@app.get("/")
def root():
    return {"status": "online", "server": "Google Colab 12GB RAM", "ready": True}

@app.post("/api/render")
def render_short(req: RenderRequest):
    job_id = str(uuid.uuid4())[:8]
    raw_video = f"/content/{job_id}_raw.mp4"
    output_video = f"{OUTPUT_DIR}/{job_id}_916.mp4"
    duration = max(1.0, req.end_seconds - req.start_seconds)

    # Download segment with yt-dlp
    dl_cmd = [
        "yt-dlp",
        "--format", "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best",
        "--download-sections", f"*{req.start_seconds}-{req.end_seconds}",
        "--force-keyframes-at-cuts",
        "-o", raw_video,
        req.url.strip()
    ]
    try:
        subprocess.run(dl_cmd, check=True, timeout=120)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Download failed: {str(e)}")

    # Crop 9:16 & burn hook banner
    vf = "crop=ih*9/16:ih:(iw-ow)/2:0,scale=1080:1920:flags=lanczos"
    if req.hook_text and req.hook_text.strip():
        clean_txt = re.sub(r"['\":;]", "", req.hook_text.strip())
        vf += f",drawbox=y=160:color=black@0.65:width=iw:height=130:t=fill,drawtext=text='{clean_txt}':fontcolor=white:fontsize=48:x=(w-text_w)/2:y=200"

    ff_cmd = [
        "ffmpeg", "-y",
        "-i", raw_video,
        "-vf", vf,
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "22",
        "-c:a", "aac",
        "-b:a", "128k",
        "-t", str(duration),
        output_video
    ]
    try:
        subprocess.run(ff_cmd, check=True, timeout=180)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Encoding failed: {str(e)}")
    finally:
        if os.path.exists(raw_video):
            os.remove(raw_video)

    return {"status": "success", "job_id": job_id, "download_url": f"/downloads/{job_id}_916.mp4"}

@app.get("/downloads/{filename}")
def get_download(filename: str):
    p = os.path.join(OUTPUT_DIR, filename)
    if not os.path.exists(p):
        raise HTTPException(status_code=404, detail="File not found")
    return FileResponse(p, media_type="video/mp4", filename=filename)
"""

with open("/content/colab_app.py", "w") as f:
    f.write(server_code)

# 3. Start server in background
import subprocess, time, re
subprocess.Popen(["python", "-m", "uvicorn", "colab_app:app", "--host", "127.0.0.1", "--port", "8000"])
time.sleep(3)

# 4. Start Cloudflare Tunnel and display public URL
print("\n🚀 Starting Cloudflare Tunnel...")
cf_proc = subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in cf_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        print("\n" + "="*60)
        print("🎉 YOUR COLAB 12GB VIDEO BACKEND IS ONLINE!")
        print(f"👉 COPY THIS URL:  {tunnel_url}")
        print("="*60 + "\n")
        print("Now paste this URL into your ViralClip Studio (Settings -> Colab Worker URL)!")
        break

cf_proc.wait()
